# Final — LangGraph Orchestration: Router → Planner → Retriever → Answerer/Critic

**Deliverable:** Final — LangGraph multi-agent orchestration

`prompts/` has been the whole team's Prompt-Disclosure artifact since Checkpoint work began (`prompts/README.md`), but until now nothing actually *loaded* those files into a running agent graph — the gap called out in `README_shane.md`: "prompts/ exists but nothing loads them into real running code." This notebook closes that gap.

It wires together:

* `src/rag/llm.py` — the single `call_llm()` call-site (Anthropic, with a graceful MockLLM fallback so this notebook runs with **zero API keys**).
* `src/rag/nodes.py` — one function per node (`router_node`, `planner_node`, `retriever_node`, `answerer_critic_node`), each reading its prompt file(s) fresh from disk on every call.
* `src/rag/graph.py` — `build_graph()`, the single source of truth for the compiled LangGraph `StateGraph`.

Along the way we also confirm a stale-value fix: `rag.config.Config.llm_model` used to default to the retired alias `claude-3-5-sonnet-latest`; it now defaults to `claude-sonnet-5`.

## 0. Setup

Two `sys.path` entries are needed: `../src` (this subtree's own package) and the **sibling** `web-search-mcp/` directory, because `rag.nodes.retriever_node` imports `web_search()` from there directly (not through the MCP protocol) — the same sibling-directory relative-path pattern `web-search-mcp/combined_mcp_server.py` uses to import `rag.rag_search` from *this* subtree, and for the identical reason: `rag.config`'s bare `load_dotenv()` only walks **upward** from the current working directory, so a sibling directory's `.env` is never found by accident — it has to be loaded by explicit path. We do the same here for the repo root's `.env`.

In [1]:
import os, sys
sys.path.append(os.path.abspath('../src'))
sys.path.append(os.path.abspath('../web-search-mcp'))

from dotenv import load_dotenv
load_dotenv(os.path.abspath('../.env'))  # explicit path -- see note above

from rag.config import get_config

cfg = get_config(refresh=True)

# .env's data paths (RAW_CSV/PROCESSED_DIR/INDEX_DIR) are written as
# relative strings meant to be resolved against the repo root as CWD (same
# caveat combined_mcp_server.py documents for vectorstore.py). nbconvert
# runs a notebook's kernel with the notebook's OWN directory as CWD
# (notebooks/), not the repo root -- so anchor them explicitly to this
# notebook's parent directory rather than relying on CWD.
_REPO_ROOT = os.path.abspath('..')
cfg.index_dir = os.path.join(_REPO_ROOT, cfg.index_dir)
cfg.processed_dir = os.path.join(_REPO_ROOT, cfg.processed_dir)
cfg.raw_csv = os.path.join(_REPO_ROOT, cfg.raw_csv)

print('llm_provider :', cfg.llm_provider)
print('llm_model    :', cfg.llm_model)
assert cfg.llm_model == 'claude-sonnet-5', (
    'stale LLM_MODEL default not fixed -- expected claude-sonnet-5, got ' + cfg.llm_model
)
print('confirmed: llm_model is claude-sonnet-5, not the retired claude-3-5-sonnet-latest alias')

llm_provider : anthropic
llm_model    : claude-sonnet-5
confirmed: llm_model is claude-sonnet-5, not the retired claude-3-5-sonnet-latest alias


## 0b. API key check (Colab-friendly)

Same `google.colab.userdata` → `getpass` fallback pattern as `notebooks/colab_end_to_end_demo.ipynb` §2b, applied to `ANTHROPIC_API_KEY`. Paste a key to see `rag.llm.call_llm` make a real Anthropic call; leave it blank and everything below still runs end to end against the MockLLM fallback in `src/rag/llm.py`.

In [2]:
from getpass import getpass

def _get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return ''

# Try Colab secrets first (silent); only prompt interactively if none is set.
anthropic_key = _get_secret('ANTHROPIC_API_KEY') or os.environ.get('ANTHROPIC_API_KEY', '')
if not anthropic_key:
    try:
        entered = getpass('ANTHROPIC_API_KEY (or Enter to skip -- MockLLM fallback): ')
    except Exception:
        # No interactive stdin available (e.g. a non-interactive "run all"
        # outside Colab's own frontend) -- skip rather than block.
        entered = ''
    if entered:
        anthropic_key = entered

if anthropic_key:
    os.environ['ANTHROPIC_API_KEY'] = anthropic_key

HAVE_LLM_KEY = bool(anthropic_key)
print('HAVE_LLM_KEY:', HAVE_LLM_KEY)

HAVE_LLM_KEY: False


## 1. Load all prompts from disk (Prompt Disclosure → real code)

Every file below is read from `prompts/` at run time -- exactly what `rag.nodes` does internally (via `PROMPTS_DIR = Path(__file__).resolve().parents[2] / "prompts"`, mirroring `rag.config.REPO_ROOT`'s own resolution pattern). Printing length + a preview here proves there is no hardcoded prompt text anywhere in `nodes.py` -- disclosure really does equal what runs.

In [3]:
import json
from pathlib import Path
from rag.nodes import PROMPTS_DIR, FEWSHOTS_DIR

prompt_files = [
    'system_assistant.md',
    'router_intent.md',
    'planner.md',
    'retriever_tool_instructions.md',
    'answerer_critic.md',
]
fewshot_files = [
    'router_examples.json',
    'planner_examples.json',
    'answerer_examples.json',
]

for name in prompt_files:
    text = (PROMPTS_DIR / name).read_text()
    print(f'{name:35s} {len(text):5d} chars | {text.strip().splitlines()[0][:70]}')

print()
for name in fewshot_files:
    data = json.loads((FEWSHOTS_DIR / name).read_text())
    print(f'{name:35s} {len(data):3d} examples')

system_assistant.md                  1912 chars | # System Prompt — Global Assistant Persona & Rules
router_intent.md                     1680 chars | # Router / Intent-Classifier Prompt
planner.md                           1808 chars | # Planner Prompt (rubric)
retriever_tool_instructions.md       1780 chars | # Retriever — Tool-Use Instructions
answerer_critic.md                   1900 chars | # Answerer + Critic Prompt

router_examples.json                  4 examples
planner_examples.json                 2 examples
answerer_examples.json                2 examples


## 2. Shared graph state

`rag.graph.GraphState` is the single `TypedDict` threaded through every node -- the fields each node reads/writes.

In [4]:
from rag.graph import GraphState

print('GraphState fields:')
for field_name, field_type in GraphState.__annotations__.items():
    print(f'  {field_name:16s} {field_type}')

GraphState fields:
  transcript       ForwardRef('str', module='rag.graph')
  router_output    ForwardRef('dict', module='rag.graph')
  planner_output   ForwardRef('dict', module='rag.graph')
  rag_results      ForwardRef('dict', module='rag.graph')
  web_results      ForwardRef('dict', module='rag.graph')
  reconciled       ForwardRef('dict', module='rag.graph')
  answerer_output  ForwardRef('dict', module='rag.graph')
  critic_output    ForwardRef('dict', module='rag.graph')
  revise_count     ForwardRef('int', module='rag.graph')


## 3. LLM call helper

`rag.llm.call_llm(system, user, model=None, max_tokens=1024, mock_response=None)` is the one function every node goes through. It dispatches on `get_config().llm_provider` (model-agnostic / config-driven, per `config.py`'s own doc comment) and only the `"anthropic"` path is fully implemented for now -- other providers raise `NotImplementedError`, per the plan. If `ANTHROPIC_API_KEY` is unset (or the real call fails for any reason), it never raises: it degrades to a `MockLLM` fallback and logs which path was taken to stderr, matching the graceful-degradation pattern already used by `web-search-mcp/web_search.py` and `rag/tts.py`.

In [5]:
from rag.llm import call_llm

test_response = call_llm(
    system='You are a test.',
    user='Say hello in one word.',
    mock_response={'mock': True, 'note': 'this is the offline fallback path'},
)
print('call_llm() response:', test_response)

call_llm() response: {"mock": true, "note": "this is the offline fallback path"}


llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)


## 4. Router node

`router_node(state, llm_fn=call_llm)` builds `system_assistant.md + router_intent.md` as the system prompt, includes `router_examples.json` as few-shot context, calls the LLM, and parses the response against `router_intent.md`'s documented schema (`task`, `constraints`, `keywords`, `safety_flags`) -- raising a clear error on malformed output rather than swallowing it. We run it on two sample transcripts from `router_examples.json`, including the safety-flag example ("mix bleach and ammonia").

In [6]:
from rag.nodes import router_node

DEMO_TRANSCRIPT = 'Recommend an eco-friendly stainless-steel cleaner under fifteen dollars.'
SAFETY_TRANSCRIPT = 'Can I mix bleach and ammonia to clean my grout faster?'

router_state = router_node({'transcript': DEMO_TRANSCRIPT})
safety_state = router_node({'transcript': SAFETY_TRANSCRIPT})

llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)
llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)


In [7]:
print('--- demo transcript router_output ---')
print(json.dumps(router_state['router_output'], indent=2))
print()
print('--- safety-flag transcript router_output ---')
print(json.dumps(safety_state['router_output'], indent=2))
assert 'safety_flags' in safety_state['router_output']

--- demo transcript router_output ---
{
  "task": "product_recommendation",
  "constraints": {
    "price_max": null,
    "price_min": null,
    "material": "stainless steel",
    "brand": null,
    "eco_preference": true,
    "min_rating": null,
    "wants_live": false
  },
  "keywords": [
    "Recommend",
    "eco-friendly",
    "stainless-steel",
    "cleaner",
    "under"
  ],
  "safety_flags": []
}

--- safety-flag transcript router_output ---
{
  "task": "product_recommendation",
  "constraints": {
    "price_max": null,
    "price_min": null,
    "material": null,
    "brand": null,
    "eco_preference": false,
    "min_rating": null,
    "wants_live": false
  },
  "keywords": [
    "bleach",
    "ammonia",
    "clean",
    "grout",
    "faster"
  ],
  "safety_flags": [
    "mixing_chemicals"
  ]
}


## 5. Planner node

`planner_node(state, llm_fn=call_llm)` builds `planner.md` + `planner_examples.json` as few-shot context, takes `state["router_output"]` as input, and produces the plan JSON documented in `planner.md`'s "Output schema" (`sources`, `call_web_search`, `filters`, `query`, `comparison_criteria`, `k`, `reconcile_on`). We feed it the demo transcript's Router output from the previous cell.

In [8]:
from rag.nodes import planner_node

planner_state = planner_node(router_state)

llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)


In [9]:
print(json.dumps(planner_state['planner_output'], indent=2))

{
  "sources": [
    "rag.search"
  ],
  "call_web_search": false,
  "filters": {
    "material": "stainless steel"
  },
  "query": "Recommend eco-friendly stainless-steel cleaner under",
  "comparison_criteria": [
    "price",
    "rating",
    "price_per_oz",
    "ingredients"
  ],
  "k": 3,
  "reconcile_on": []
}


## 6. Retriever node

`retriever_node(state)` (the only `async def` node -- it may `await` `web_search()`) calls the real `rag.rag_search()` tool using the Planner's `query`/`k`/`filters`, and additionally calls `web_search()` from `web-search-mcp/web_search.py` when the Planner's `call_web_search` field is `true`. Both result sets are reconciled via `rag.reconcile.reconcile()` (private facts as the grounded baseline; discrepancies flagged, never silently overwritten).

In [10]:
from rag.nodes import retriever_node

# Jupyter supports top-level await in code cells.
retriever_state = await retriever_node(planner_state)

In [11]:
print('rag.search count :', retriever_state['rag_results']['count'])
print('web.search count :', retriever_state['web_results']['count'])
print()
for item in retriever_state['reconciled']['items']:
    flag = f" -- DISCREPANCY: {item['discrepancy']['detail']}" if item.get('discrepancy') else ''
    print(f"  {item['title']:55s} \${item['price']:<7}{flag}")
print()
print('unmatched web results:', len(retriever_state['reconciled']['unmatched_web']))

rag.search count : 3
web.search count : 0

  NatureNest Stainless Steel Cleaner, 8 oz travel         \$6.49   
  Steel-Safe Eco Stainless Steel Cleaner & Polish, 16 oz  \$12.49  
  EverGreen Stainless Steel Wipes Refill, 40 ct           \$13.49  

unmatched web results: 0


<>:6: SyntaxWarning: invalid escape sequence '\$'
<>:6: SyntaxWarning: invalid escape sequence '\$'
/var/folders/47/q5r4vm094rs50jyk76tdz7fr0000gn/T/ipykernel_97586/3530765245.py:6: SyntaxWarning: invalid escape sequence '\$'
  print(f"  {item['title']:55s} \${item['price']:<7}{flag}")


## 7. Answerer/Critic node

`answerer_critic_node(state, llm_fn=call_llm)` builds `answerer_critic.md` + `answerer_examples.json`, calls the LLM once for the **Answerer** role (schema: `speech`, `citations`, `comparison_table` -- ≤15s spoken, every claim traces to a `doc_id`/`url`), then again for the **Critic** role (schema: `grounded`, `unsafe`, `reasons`, `action`). Per `answerer_critic.md`, on `action: "revise"` the Answerer regenerates **once** -- this is bounded in `graph.py` via `state["revise_count"]` and a conditional edge, not by looping inside the node itself.

In [12]:
from rag.nodes import answerer_critic_node

answerer_state = answerer_critic_node(retriever_state)

llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)
llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)


In [13]:
print('--- Answerer payload ---')
print(json.dumps(answerer_state['answerer_output'], indent=2))
print()
print('--- Critic verdict ---')
print(json.dumps(answerer_state['critic_output'], indent=2))

--- Answerer payload ---
{
  "speech": "My top pick is NatureNest Stainless Steel Cleaner, 8 oz travel at $6.49, 4.4 stars. I compared it with 2 alternatives \u2014 details and sources are on your screen. Want the most affordable or the highest rated?",
  "citations": [
    {
      "doc_id": "b5bc95c5d69c5de186be4a4ef1a98ed1",
      "title": "NatureNest Stainless Steel Cleaner, 8 oz travel",
      "url": "https://www.amazon.com/dp/B0SAMPLE018",
      "source": "private"
    }
  ],
  "comparison_table": [
    {
      "title": "NatureNest Stainless Steel Cleaner, 8 oz travel",
      "price": 6.49,
      "rating": 4.4,
      "price_per_oz": 0.8113,
      "ingredients": "Water, Coco-glucoside, Citric acid, Sodium gluconate, Peppermint oil",
      "doc_id": "b5bc95c5d69c5de186be4a4ef1a98ed1"
    }
  ]
}

--- Critic verdict ---
{
  "grounded": true,
  "unsafe": false,
  "reasons": [],
  "action": "accept"
}


## 8. Wire the StateGraph

`rag.graph.build_graph()` is the single source of truth other code (a future Streamlit UI, the eval harness) should import rather than re-wiring the graph elsewhere. Entry point `router` → `planner` → `retriever` → `answerer_critic` → conditional edge (`revise` → back to `answerer_critic`, `accept`/end → `END`).

In [14]:
from rag.graph import build_graph

compiled_graph = build_graph()
print(compiled_graph)

## 9. End-to-end run

One call through the whole compiled graph on a real transcript -- the same demo query used throughout the repo (`README_shane.md`'s "demo query", `router_examples.json`'s first example): *"Recommend an eco-friendly stainless-steel cleaner under fifteen dollars."* Because the graph contains an async node (`retriever`), we invoke it via `.ainvoke()` (LangGraph requires the async entry point on any graph with an async node, even from a notebook that could otherwise call `.invoke()`).

In [15]:
final_state = await compiled_graph.ainvoke({'transcript': DEMO_TRANSCRIPT})

print('SPOKEN ANSWER:')
print(' ', final_state['answerer_output']['speech'])
print()
print('CRITIC VERDICT:', final_state['critic_output']['action'],
      '(grounded=' + str(final_state['critic_output']['grounded']) + ',',
      'unsafe=' + str(final_state['critic_output']['unsafe']) + ')')
print()
print('CITATIONS:')
for c in final_state['answerer_output']['citations']:
    print('  -', c['title'], '|', c.get('doc_id') or c.get('url'))

SPOKEN ANSWER:
  My top pick is NatureNest Stainless Steel Cleaner, 8 oz travel at $6.49, 4.4 stars. I compared it with 2 alternatives — details and sources are on your screen. Want the most affordable or the highest rated?

CRITIC VERDICT: accept (grounded=True, unsafe=False)

CITATIONS:
  - NatureNest Stainless Steel Cleaner, 8 oz travel | b5bc95c5d69c5de186be4a4ef1a98ed1


llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)
llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)
llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)
llm.call_llm: ANTHROPIC_API_KEY is not set; falling back to MockLLM so the graph still runs offline.
llm.call_llm: using MockLLM (no real API call made)


## 10. (Optional) Speak the final answer

Soft-imports `rag.tts.speak()` (built separately -- see `notebooks/03_tts_summary.ipynb`) wrapped in try/except, so this cell degrades gracefully rather than failing the notebook if `tts.py` or its offline `pyttsx3` dependency isn't available in this environment.

In [16]:
try:
    from rag.tts import speak
    audio_path = speak(final_state['answerer_output']['speech'])
    print('wrote', audio_path)
    try:
        from IPython.display import Audio, display
        display(Audio(str(audio_path)))
    except Exception:
        pass
except Exception as e:
    print(f'tts unavailable in this environment ({e!r}); skipping audio playback.')

wrote /Users/ceverson/Development/Academic/ADSP_32028/final/audio/summary_564423f8d12b.wav


---
### Handoff notes

**What was built:** `src/rag/llm.py` (`call_llm()`, the one LLM call-site), `src/rag/nodes.py` (`router_node`, `planner_node`, `retriever_node`, `answerer_critic_node` -- each loading its prompt(s) fresh from disk, each accepting an optional `llm_fn` for dependency-injected testing), and `src/rag/graph.py` (`build_graph()`, the single source of truth for the compiled `StateGraph`).

**Gap closed:** `prompts/` previously documented every prompt but nothing loaded them into running code (`README_shane.md`). This notebook and `nodes.py` are the concrete demonstration that disclosure == what runs (§1 above reads and prints every prompt file straight from disk).

**Model-default fix applied:** `rag.config.Config.llm_model` defaulted to the retired alias `claude-3-5-sonnet-latest`; fixed to `claude-sonnet-5` in `src/rag/config.py` (and mirrored in `.env.example`'s `LLM_MODEL=` line and `prompts/README.md`'s Conventions section).

**Known limitations:**
* Only the `"anthropic"` provider is implemented in `llm.py` -- `openai`/`google`/`bedrock`/`local` raise `NotImplementedError`, per the plan's scope.
* The `MockLLM` fallback shape is deliberately simple: each node supplies its own structurally-valid canned response (see `_router_mock_response` / `_planner_mock_response` / `_answerer_mock_response` / `_critic_mock_response` in `nodes.py`) rather than `call_llm` sniffing prompt text for schema markers -- simpler and independently testable, per the plan.
* The Critic's mock response always accepts (`action: "accept"`), so the revise loop is only exercised with a real LLM key or an injected `llm_fn` in tests (`tests/test_nodes.py::test_answerer_critic_node_revise_path_increments_revise_count`).
* `retriever_node`'s `web.search` path inherits `web_search.py`'s own documented limitation: `price`/`availability` are always `None` from the current providers, so most reconciliation discrepancy flags won't fire until that's filled in.